## 1) Load packages 

In [1]:
import json
import os 
import pandas as pd
import glob
from tqdm import tqdm 
from collections import defaultdict
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import inchi

## 2) Functions 

In [2]:
def parse_mibig_json(json_file):
    """
    Parses a MIBiG JSON file and extracts compound metadata,
    prioritizing one compound as the main based on bioactivity and database presence.

    Parameters:
        json_file (str): Path to the MIBiG JSON file.

    Returns:
        dict: A dictionary containing compound details and metadata.
    """
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Initialize output dictionary
    result = {
        'accession': data.get('accession', ''),
        'compound_names': [],
        'structures': [],
        'masses': [],
        'formulas': [],
        'main_compound': None,
        'main_structure': None,
        'main_mass': None,
        'main_formula': None
    }
    
    # Extract all compounds
    compounds = data.get('compounds', [])
    for compound in compounds:
        result['compound_names'].append(compound.get('name', ''))
        result['structures'].append(compound.get('structure', ''))
        result['masses'].append(compound.get('mass', ''))
        result['formulas'].append(compound.get('formula', ''))
    
    # Determine main compound using priority rules
    if compounds:
        # Rule 1: Prefer compounds with bioactivity evidence
        bioactive = [c for c in compounds if c.get('bioactivities')]
        if bioactive:
            main_compound = bioactive[0]
        else:
            # Rule 2: Prefer compounds with database IDs (PubChem, ChemBL, etc.)
            db_compounds = [c for c in compounds if c.get('databaseIds')]
            if db_compounds:
                main_compound = db_compounds[0]
            else:
                # Rule 3: Just take the first compound
                main_compound = compounds[0]
        
        result['main_compound'] = main_compound.get('name', '')
        result['main_structure'] = main_compound.get('structure', '')
        result['main_mass'] = main_compound.get('mass', '')
        result['main_formula'] = main_compound.get('formula', '')
    
    return result

In [3]:
def process_gnps_json(json_file):
    """
    Processes a GNPS-style JSON file, selecting the best version of each spectrum
    based on heuristic scoring (SMILES, InChIKey, timestamps, and Precursor_MZ).

    Parameters:
        json_file (str): Path to the GNPS JSON dataset.

    Returns:
        pd.DataFrame: A DataFrame containing the best spectrum entries by ID.
    """
    # Load the JSON data with progress bar (for large files it'd be better to proceed entry by entry)
    print("Loading JSON file...")
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Dictionary to store the best version of each spectrum
    spectra = defaultdict(list)
    
    # Collect all entries grouped by spectrum_id with progress bar
    print("Grouping spectra by ID...")
    for entry in tqdm(data, desc="Processing entries"):
        spectra[entry['spectrum_id']].append(entry)
    
    processed_data = []
    
    # Process each spectrum with progress bar
    print("Selecting best entries...")
    for spectrum_id, entries in tqdm(spectra.items(), desc="Processing spectra"):
        # Selection criteria: prefer entries with more complete information
        best_entry = None
        best_score = -1
        
        for entry in entries:
            score = 0
            # Prioritize entries with SMILES
            if entry.get('Smiles') and entry['Smiles'] != 'N/A':
                score += 2
            # Prioritize entries with InChIKey
            if entry.get('InChIKey_smiles') and entry['InChIKey_smiles'] != '':
                score += 1
            # Prefer newer entries (higher create_time)
            if 'create_time' in entry:
                score += 0.5  # Small bonus for having timestamp
            
            if score > best_score:
                best_score = score
                best_entry = entry
            elif score == best_score:
                # If scores are equal, prefer the one with Precursor_MZ
                if 'Precursor_MZ' in best_entry and 'Precursor_MZ' in entry:
                    if float(entry['Precursor_MZ']) > float(best_entry['Precursor_MZ']):
                        best_entry = entry
        
        if best_entry is None:
            best_entry = entries[0]  # fallback to first entry if no selection
        
        # Extract the desired fields
        # Return the value if existed, otherwise return default value 
        processed_entry = {
            'spectrum_id': spectrum_id,
            'Compound_Name': best_entry.get('Compound_Name', 'N/A'),
            'Precursor_MZ': best_entry.get('Precursor_MZ', '0.0'),
            'ExactMass': best_entry.get('ExactMass', '0.0'),
            'Smiles': best_entry.get('Smiles', 'N/A'),
            'INCHI': best_entry.get('INCHI', 'N/A'),
            'InChIKey_smiles': best_entry.get('InChIKey_smiles', 'N/A')
        }
        processed_data.append(processed_entry)
    
    return pd.DataFrame(processed_data)

In [4]:
# Example usage 
# root path to the mibig json files 
root_path = '/home/fab25/datasets/mibig/mibig_json_4.0'
# Example usage for one file
parsed_data = parse_mibig_json(os.path.join(root_path,'BGC0001001.json'))
print(parsed_data)

{'accession': 'BGC0001001', 'compound_names': ['jamaicamide A', 'jamaicamide B', 'jamaicamide C'], 'structures': ['CO\\C(CCNC(=O)CC\\C=C\\C(C)CC\\C(CCCC#CBr)=C\\Cl)=C\\C(=O)N1C(C)C=CC1=O', 'CO\\C(CCNC(=O)CC\\C=C\\C(C)CC\\C(CCCC#C)=C\\Cl)=C\\C(=O)N1C(C)C=CC1=O', 'CO\\C(CCNC(=O)CC\\C=C\\C(C)CC\\C(CCCC=C)=C\\Cl)=C\\C(=O)N1C(C)C=CC1=O'], 'masses': [566.1546974120001, 488.244185344, 490.259835408], 'formulas': ['C27H36BrClN2O4', 'C27H37ClN2O4', 'C27H39ClN2O4'], 'main_compound': 'jamaicamide A', 'main_structure': 'CO\\C(CCNC(=O)CC\\C=C\\C(C)CC\\C(CCCC#CBr)=C\\Cl)=C\\C(=O)N1C(C)C=CC1=O', 'main_mass': 566.1546974120001, 'main_formula': 'C27H36BrClN2O4'}


## 3) Extract data from json files 
### mibig 

In [ ]:
# all_data = []   # List of dictionaries to hold all parsed data 
# for json_file in tqdm(glob.glob(os.path.join(root_path,'*.json'))):
#     all_data.append(parse_mibig_json(json_file))

# # Create DataFrame
# df = pd.DataFrame(all_data)

# # export path
# export_dir = '/home/fab25/datasets/mibig'
# # Save to CSV
# df.to_csv(os.path.join(export_dir, 'mibig_compounds.csv'), index=False)

100%|██████████| 3013/3013 [00:01<00:00, 1763.30it/s]


### gnps 

In [ ]:
# root_gnps_path = "/home/fab25/datasets/gnps"
# # Example usage
# df = process_gnps_json(os.path.join(root_gnps_path, 'ALL_GNPS.json'))

# # Save to CSV
# df.to_csv(os.path.join(root_gnps_path, 'gnps_spectra_processed.csv'), index=False)

# # Display sample
# print(df.head())

Loading JSON file...
Grouping spectra by ID...


Processing entries: 100%|██████████| 1299134/1299134 [00:09<00:00, 131328.75it/s]


Selecting best entries...


Processing spectra: 100%|██████████| 1299134/1299134 [00:04<00:00, 314305.57it/s]


          spectrum_id          Compound_Name Precursor_MZ ExactMass  \
0  CCMSLIB00000001547  3-Des-Microcystein_LR       981.54       0.0   
1  CCMSLIB00000001548             Hoiamide B       940.25   939.452   
2  CCMSLIB00000001549          Malyngamide C        456.1   455.244   
3  CCMSLIB00000001550             Scytonemin        545.0       0.0   
4  CCMSLIB00000001551      Salinisporamide A      314.116   313.108   

                                              Smiles  \
0  CC(C)CC1NC(=O)C(C)NC(=O)C(=C)N(C)C(=O)CCC(NC(=...   
1  CCC[C@@H](C)[C@@H]([C@H](C)[C@@H]1[C@H]([C@H](...   
2  CCCCCCC[C@@H](C/C=C/CCC(=O)NC/C(=C/Cl)/[C@@]12...   
3  OC1=CC=C(\C=C2\C(=O)C(C3=C4C5=C(C=CC=C5)N=C4\C...   
4                                                N/A   

                                               INCHI InChIKey_smiles  
0                                                N/A                  
1  InChI=1S/C45H73N5O10S3/c1-14-17-24(6)34(52)26(...                  
2  InChI=1S/C24H38ClNO5

## 4) Combine MiBIG and GNPS metadata files 

In [27]:
export_dir = '/home/fab25/datasets/mibig'
mibig_df = pd.read_csv(os.path.join(export_dir, 'mibig_compounds.csv'))
print(f'Shape of mibig df: {mibig_df.shape}')
mibig_df.head()

Shape of mibig df: (3013, 9)


,accession,compound_names,structures,masses,formulas,main_compound,main_structure,main_mass,main_formula
0,BGC0000001,"['abyssomicin C', 'atrop-abyssomicin C']",['CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\\C=C/C(=O)[C@...,"[346.141638424, 346.141638424]","['C19H22O6', 'C19H22O6']",abyssomicin C,CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H]...,346.141638,C19H22O6
1,BGC0000002,['aculeximycin'],['CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(...,[1672.9651350679992],['C81H144N2O33'],aculeximycin,CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C)...,1672.965135,C81H144N2O33
2,BGC0000003,['AF-toxin'],['CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C...,[440.20463260399987],['C22H32O9'],AF-toxin,CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C)O...,440.204633,C22H32O9
3,BGC0000004,['aflatoxin G1'],['COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[...,[328.058302724],['C17H12O7'],aflatoxin G1,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[C@...,328.058303,C17H12O7
4,BGC0000005,['aflatoxin'],['COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4...,[328.058302724],['C17H12O7'],aflatoxin,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4=C1,328.058303,C17H12O7


In [6]:
# Load the GNPS dataset 
root_gnps_path = "/home/fab25/datasets/gnps"
gnps_df = pd.read_csv(os.path.join(root_gnps_path, 'gnps_spectra_processed.csv'))
print(f'shape of gnps df:{gnps_df.shape}')
gnps_df.head()

shape of gnps df:(1299134, 7)


,spectrum_id,Compound_Name,Precursor_MZ,ExactMass,Smiles,INCHI,InChIKey_smiles
0,CCMSLIB00000001547,3-Des-Microcystein_LR,981.540,0.000,CC(C)CC1NC(=O)C(C)NC(=O)C(=C)N(C)C(=O)CCC(NC(=...,NaN,NaN
1,CCMSLIB00000001548,Hoiamide B,940.250,939.452,CCC[C@@H](C)[C@@H]([C@H](C)[C@@H]1[C@H]([C@H](...,InChI=1S/C45H73N5O10S3/c1-14-17-24(6)34(52)26(...,NaN
2,CCMSLIB00000001549,Malyngamide C,456.100,455.244,CCCCCCC[C@@H](C/C=C/CCC(=O)NC/C(=C/Cl)/[C@@]12...,InChI=1S/C24H38ClNO5/c1-3-4-5-6-8-11-19(30-2)1...,NaN
3,CCMSLIB00000001550,Scytonemin,545.000,0.000,OC1=CC=C(\C=C2\C(=O)C(C3=C4C5=C(C=CC=C5)N=C4\C...,InChI=1S/C36H20N2O4/c39-21-13-9-19(10-14-21)17...,NaN
4,CCMSLIB00000001551,Salinisporamide A,314.116,313.108,NaN,NaN,NaN


In [7]:
# Number of available InChIKey_smiles
print(f'Number of NaN InChIKey_smiles in gnps_df: {gnps_df.InChIKey_smiles.isna().shape[0]}')

Number of NaN InChIKey_smiles in gnps_df: 1299134


In [9]:
print(f'Number of unique InChIKey_smiles (BGCs) in mibig_df: {mibig_df.main_compound.value_counts().shape[0]}')

Number of unique InChIKey_smiles (BGCs) in mibig_df: 2563


#### 4.1) Add InChIKey connectivity layer to MIBiG data using RDKit

- Method 1

In [28]:
def smiles_to_inchikey_connectivity(smiles):
    """
    Converts a SMILES string to its full InChIKey and connectivity layer.

    Returns:
        Tuple[str, str]: (Full InChIKey, Connectivity layer)
        or None if the SMILES is invalid.
    """
    try:
        mol = Chem.MolFromSmiles(smiles)
        if not mol:
            return None
        inchikey = inchi.MolToInchiKey(mol)
        return inchikey, inchikey.split('-')[0]
    except Exception as e:
        print(f"Error processing SMILES '{smiles}': {e}")
        return None

In [29]:
# Apply the function and convert the result to two separate columns
mibig_df[['InChIKey_smiles', 'inchikey_connectivity']] = mibig_df['main_structure'].apply(
    lambda smi: pd.Series(smiles_to_inchikey_connectivity(smi)) if pd.notna(smi) else pd.Series([None, None]))

[23:56:46] Conflicting single bond directions around double bond at index 26.
[23:56:46]   BondStereo set to STEREONONE and single bond directions set to NONE.
[23:56:47] WARNING: not removing hydrogen atom without neighbors
[23:56:47] WARNING: not removing hydrogen atom without neighbors
[23:56:47] WARNING: not removing hydrogen atom without neighbors
[23:56:47] WARNING: not removing hydrogen atom without neighbors


In [30]:
mibig_df.head(5)

,accession,compound_names,structures,masses,formulas,main_compound,main_structure,main_mass,main_formula,InChIKey_smiles,inchikey_connectivity
0,BGC0000001,"['abyssomicin C', 'atrop-abyssomicin C']",['CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\\C=C/C(=O)[C@...,"[346.141638424, 346.141638424]","['C19H22O6', 'C19H22O6']",abyssomicin C,CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H]...,346.141638,C19H22O6,FNEADFUPWHAVTA-PPDYZMKSSA-N,FNEADFUPWHAVTA
1,BGC0000002,['aculeximycin'],['CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(...,[1672.9651350679992],['C81H144N2O33'],aculeximycin,CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C)...,1672.965135,C81H144N2O33,VJKZKLDZOAFAEE-QIESNYARSA-N,VJKZKLDZOAFAEE
2,BGC0000003,['AF-toxin'],['CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C...,[440.20463260399987],['C22H32O9'],AF-toxin,CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C)O...,440.204633,C22H32O9,ONOBRFRRMLDPES-BGSVYHRFSA-N,ONOBRFRRMLDPES
3,BGC0000004,['aflatoxin G1'],['COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[...,[328.058302724],['C17H12O7'],aflatoxin G1,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[C@...,328.058303,C17H12O7,XWIYFDMXXLINPU-RBHXEPJQSA-N,XWIYFDMXXLINPU
4,BGC0000005,['aflatoxin'],['COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4...,[328.058302724],['C17H12O7'],aflatoxin,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4=C1,328.058303,C17H12O7,XWIYFDMXXLINPU-UHFFFAOYSA-N,XWIYFDMXXLINPU


In [31]:
def smiles_list_to_inchikeys(smiles_list):
    """
    Converts a list of SMILES strings to their InChIKeys and connectivity layers.
    Returns:
        - inchikeys: list of InChIKeys (or None if invalid)
        - connectivity_layers: list of connectivity layers (or None if invalid)
    """
    inchikeys = []
    connectivity_layers = []
    if not isinstance(smiles_list, list):
        return [None], [None]
    for smiles in smiles_list:
        try:
            mol = Chem.MolFromSmiles(smiles)
            if not mol:
                inchikeys.append(None)
                connectivity_layers.append(None)
                continue
            ik = inchi.MolToInchiKey(mol)
            inchikeys.append(ik)
            connectivity_layers.append(ik.split('-')[0])
        except Exception as e:
            print(f"Error processing SMILES '{smiles}': {e}")
            inchikeys.append(None)
            connectivity_layers.append(None)
    return inchikeys, connectivity_layers

In [53]:
from ast import literal_eval

# Ensure 'structures' is a list
mibig_df['structures_list'] = mibig_df['structures'].apply(
    lambda x: literal_eval(x) if isinstance(x, str) else x
)

# Apply the function
mibig_df[['inchikeys', 'connectivity_layers']] = mibig_df['structures_list'].apply(
    lambda smi_list: pd.Series(smiles_list_to_inchikeys(smi_list))
)

[00:09:29] Invalid InChI prefix in generating InChI Key
[00:09:29] Invalid InChI prefix in generating InChI Key
[00:09:29] Invalid InChI prefix in generating InChI Key
[00:09:29] Invalid InChI prefix in generating InChI Key
[00:09:29] Invalid InChI prefix in generating InChI Key
[00:09:29] Invalid InChI prefix in generating InChI Key
[00:09:29] Conflicting single bond directions around double bond at index 26.
[00:09:29]   BondStereo set to STEREONONE and single bond directions set to NONE.
[00:09:29] Invalid InChI prefix in generating InChI Key
[00:09:30] Invalid InChI prefix in generating InChI Key
[00:09:30] Invalid InChI prefix in generating InChI Key
[00:09:30] Invalid InChI prefix in generating InChI Key
[00:09:30] Invalid InChI prefix in generating InChI Key
[00:09:30] Invalid InChI prefix in generating InChI Key
[00:09:30] Invalid InChI prefix in generating InChI Key
[00:09:30] Invalid InChI prefix in generating InChI Key
[00:09:30] Invalid InChI prefix in generating InChI Key


In [54]:
mibig_df.head(5)

,accession,compound_names,structures,masses,formulas,main_compound,main_structure,main_mass,main_formula,InChIKey_smiles,inchikey_connectivity,inchikeys,connectivity_layers,structures_list
0,BGC0000001,"['abyssomicin C', 'atrop-abyssomicin C']",['CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\\C=C/C(=O)[C@...,"[346.141638424, 346.141638424]","['C19H22O6', 'C19H22O6']",abyssomicin C,CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H]...,346.141638,C19H22O6,FNEADFUPWHAVTA-PPDYZMKSSA-N,FNEADFUPWHAVTA,"[FNEADFUPWHAVTA-PPDYZMKSSA-N, FNEADFUPWHAVTA-P...","[FNEADFUPWHAVTA, FNEADFUPWHAVTA]",[CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H...
1,BGC0000002,['aculeximycin'],['CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(...,[1672.9651350679992],['C81H144N2O33'],aculeximycin,CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C)...,1672.965135,C81H144N2O33,VJKZKLDZOAFAEE-QIESNYARSA-N,VJKZKLDZOAFAEE,[VJKZKLDZOAFAEE-QIESNYARSA-N],[VJKZKLDZOAFAEE],[CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C...
2,BGC0000003,['AF-toxin'],['CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C...,[440.20463260399987],['C22H32O9'],AF-toxin,CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C)O...,440.204633,C22H32O9,ONOBRFRRMLDPES-BGSVYHRFSA-N,ONOBRFRRMLDPES,[ONOBRFRRMLDPES-BGSVYHRFSA-N],[ONOBRFRRMLDPES],[CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C)...
3,BGC0000004,['aflatoxin G1'],['COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[...,[328.058302724],['C17H12O7'],aflatoxin G1,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[C@...,328.058303,C17H12O7,XWIYFDMXXLINPU-RBHXEPJQSA-N,XWIYFDMXXLINPU,[XWIYFDMXXLINPU-RBHXEPJQSA-N],[XWIYFDMXXLINPU],[COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[C...
4,BGC0000005,['aflatoxin'],['COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4...,[328.058302724],['C17H12O7'],aflatoxin,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4=C1,328.058303,C17H12O7,XWIYFDMXXLINPU-UHFFFAOYSA-N,XWIYFDMXXLINPU,[XWIYFDMXXLINPU-UHFFFAOYSA-N],[XWIYFDMXXLINPU],[COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4=C1]


In [78]:
# Return all unique non-None InChIKeys (or None if none), and all unique connectivity layers
def unique_or_all(lst):
    # Remove None, empty strings, and whitespace-only strings
    lst = [x for x in lst if x and str(x).strip()]
    if not lst:
        return None
    unique = list(set(lst))
    return unique if len(unique) > 1 else unique[0]

mibig_df['inchikey'] = mibig_df['inchikeys'].apply(unique_or_all)
mibig_df['connectivity_layer'] = mibig_df['connectivity_layers'].apply(unique_or_all)

In [79]:
mibig_df.head(5)

,accession,compound_names,structures,masses,formulas,main_compound,main_structure,main_mass,main_formula,InChIKey_smiles,inchikey_connectivity,inchikeys,connectivity_layers,structures_list,inchikey,connectivity_layer
0,BGC0000001,"['abyssomicin C', 'atrop-abyssomicin C']",['CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\\C=C/C(=O)[C@...,"[346.141638424, 346.141638424]","['C19H22O6', 'C19H22O6']",abyssomicin C,CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H]...,346.141638,C19H22O6,FNEADFUPWHAVTA-PPDYZMKSSA-N,FNEADFUPWHAVTA,"[FNEADFUPWHAVTA-PPDYZMKSSA-N, FNEADFUPWHAVTA-P...","[FNEADFUPWHAVTA, FNEADFUPWHAVTA]",[CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H...,"[FNEADFUPWHAVTA-PPDYZMKSSA-N, FNEADFUPWHAVTA-P...",FNEADFUPWHAVTA
1,BGC0000002,['aculeximycin'],['CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(...,[1672.9651350679992],['C81H144N2O33'],aculeximycin,CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C)...,1672.965135,C81H144N2O33,VJKZKLDZOAFAEE-QIESNYARSA-N,VJKZKLDZOAFAEE,[VJKZKLDZOAFAEE-QIESNYARSA-N],[VJKZKLDZOAFAEE],[CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C...,VJKZKLDZOAFAEE-QIESNYARSA-N,VJKZKLDZOAFAEE
2,BGC0000003,['AF-toxin'],['CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C...,[440.20463260399987],['C22H32O9'],AF-toxin,CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C)O...,440.204633,C22H32O9,ONOBRFRRMLDPES-BGSVYHRFSA-N,ONOBRFRRMLDPES,[ONOBRFRRMLDPES-BGSVYHRFSA-N],[ONOBRFRRMLDPES],[CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C)...,ONOBRFRRMLDPES-BGSVYHRFSA-N,ONOBRFRRMLDPES
3,BGC0000004,['aflatoxin G1'],['COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[...,[328.058302724],['C17H12O7'],aflatoxin G1,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[C@...,328.058303,C17H12O7,XWIYFDMXXLINPU-RBHXEPJQSA-N,XWIYFDMXXLINPU,[XWIYFDMXXLINPU-RBHXEPJQSA-N],[XWIYFDMXXLINPU],[COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[C...,XWIYFDMXXLINPU-RBHXEPJQSA-N,XWIYFDMXXLINPU
4,BGC0000005,['aflatoxin'],['COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4...,[328.058302724],['C17H12O7'],aflatoxin,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4=C1,328.058303,C17H12O7,XWIYFDMXXLINPU-UHFFFAOYSA-N,XWIYFDMXXLINPU,[XWIYFDMXXLINPU-UHFFFAOYSA-N],[XWIYFDMXXLINPU],[COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4=C1],XWIYFDMXXLINPU-UHFFFAOYSA-N,XWIYFDMXXLINPU


In [83]:
# Count if there are none in the connectivity_layer column
print(f"{mibig_df['connectivity_layer'].isna().sum()} BGCs without inchikey out of {mibig_df.shape[0]} total BGCs")

626 BGCs without inchikey out of 3013 total BGCs


In [84]:
# Count if there are none in the inchikey_connectivity column
print(f"{mibig_df['inchikey_connectivity'].isna().sum()} BGCs without inchikey out of {mibig_df.shape[0]} total BGCs")

633 BGCs without inchikey out of 3013 total BGCs


In [100]:
mibig_df.loc[mibig_df['inchikey_connectivity'].isna() & mibig_df['connectivity_layer'].notna()]

,accession,compound_names,structures,masses,formulas,main_compound,main_structure,main_mass,main_formula,InChIKey_smiles,inchikey_connectivity,inchikeys,connectivity_layers,structures_list,inchikey,connectivity_layer
1237,BGC0001270,"['gibberellin A3', 'gibberellin A1', 'gibberel...","['', '', 'C[C@@]12[C@H](C=C[C@@]3([C@@H]1[C@@H...","['', '', 330.1467238039999, 332.1623738679999,...","['', '', 'C19H22O5', 'C19H24O5', '']",gibberellin A3,NaN,NaN,NaN,None,None,"[, , SEEGHKWOBVVBTQ-NFMPGMCNSA-N, RSQSQJNRHICN...","[, , SEEGHKWOBVVBTQ, RSQSQJNRHICNNH, ]","[, , C[C@@]12[C@H](C=C[C@@]3([C@@H]1[C@@H]([C@...","[RSQSQJNRHICNNH-NFMPGMCNSA-N, SEEGHKWOBVVBTQ-N...","[RSQSQJNRHICNNH, SEEGHKWOBVVBTQ]"
1451,BGC0001492,"['abyssomicin M', 'abyssomicin N', 'abyssomici...","['', '', '', '', '', '', '', '', '', '', 'C[C@...","['', '', '', '', '', '', '', '', '', '', 394.1...","['', '', '', '', '', '', '', '', '', '', 'C20H...",abyssomicin M,NaN,NaN,NaN,None,None,"[, , , , , , , , , , FAFGDIIANZCFAM-QNPGEOKISA...","[, , , , , , , , , , FAFGDIIANZCFAM, JOZRZQMZA...","[, , , , , , , , , , C[C@H]\1C[C@](C(=O)C[C@H]...","[FAFGDIIANZCFAM-QNPGEOKISA-N, JOZRZQMZAIQZPI-M...","[JOZRZQMZAIQZPI, FAFGDIIANZCFAM]"
1549,BGC0001591,"['fatty acid enol ester', 'N-tetradecanoyl tyr...","['', 'CCCCCCCCCCCC(=O)N[C@@H](Cc1ccc(O)cc1)C(O...","['', 365.25660859999994]","['', 'C21H35NO4']",fatty acid enol ester,NaN,NaN,NaN,None,None,"[, JJNOOOPGJKEVQC-IBGZPJMESA-N]","[, JJNOOOPGJKEVQC]","[, CCCCCCCCCCCC(=O)N[C@@H](Cc1ccc(O)cc1)C(O)O]",JJNOOOPGJKEVQC-IBGZPJMESA-N,JJNOOOPGJKEVQC
1748,BGC0001801,"['thiazostatin', 'watasemycin A', 'watasemycin...","['', '[H][C@]1(SC[C@@](C)(N1C)C(O)=O)[C@]1([H]...","['', 352.09153449999997, 352.09153449999997, '...","['', 'C16H20N2O3S2', 'C16H20N2O3S2', '', '']",thiazostatin,NaN,NaN,NaN,None,None,"[, NOEXPDVJQLSPPC-UODBANLJSA-N, NOEXPDVJQLSPPC...","[, NOEXPDVJQLSPPC, NOEXPDVJQLSPPC, , ]","[, [H][C@]1(SC[C@@](C)(N1C)C(O)=O)[C@]1([H])N=...","[NOEXPDVJQLSPPC-MQQCTCJLSA-N, NOEXPDVJQLSPPC-U...",NOEXPDVJQLSPPC
2254,BGC0002326,"['endopyrrole B', 'endopyrrole C', 'endopyrrol...","['', '', 'O=C(N[C@H](C(N1CCC[C@H]1C(N[C@@H](C(...","['', '', '']","['', '', '']",endopyrrole B,NaN,NaN,NaN,None,None,"[, , KRMWEQCPKUMYBI-DKKFUSLWSA-N]","[, , KRMWEQCPKUMYBI]","[, , O=C(N[C@H](C(N1CCC[C@H]1C(N[C@@H](C(C)C)C...",KRMWEQCPKUMYBI-DKKFUSLWSA-N,KRMWEQCPKUMYBI
2368,BGC0002445,"['armillyl orsellinate', '8α-hydroxy-6-protoil...","['', 'CC1=C2CC[C@]2(C)[C@H]2CC(C)(C)C[C@H]2[C@...","['', '']","['', '']",armillyl orsellinate,NaN,NaN,NaN,None,None,"[, XTXSLQPVMVDDTB-ZRQNBYAXSA-N]","[, XTXSLQPVMVDDTB]","[, CC1=C2CC[C@]2(C)[C@H]2CC(C)(C)C[C@H]2[C@@H]1O]",XTXSLQPVMVDDTB-ZRQNBYAXSA-N,XTXSLQPVMVDDTB
2438,BGC0002517,"['tripartilactam', 'niizalactam C']","['', 'C[C@H]\\1CNC(=O)CC(=O)/C=C/C(=C/[C@@H]2[...","['', '']","['', '']",tripartilactam,NaN,NaN,NaN,None,None,"[, PUDBJLXDJHKFMZ-IXPKUMSOSA-N]","[, PUDBJLXDJHKFMZ]","[, C[C@H]\1CNC(=O)CC(=O)/C=C/C(=C/[C@@H]2[C@H]...",PUDBJLXDJHKFMZ-IXPKUMSOSA-N,PUDBJLXDJHKFMZ


In [88]:
mibig_df.head(2)

,accession,compound_names,structures,masses,formulas,main_compound,main_structure,main_mass,main_formula,InChIKey_smiles,inchikey_connectivity,inchikeys,connectivity_layers,structures_list,inchikey,connectivity_layer
0,BGC0000001,"['abyssomicin C', 'atrop-abyssomicin C']",['CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\\C=C/C(=O)[C@...,"[346.141638424, 346.141638424]","['C19H22O6', 'C19H22O6']",abyssomicin C,CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H]...,346.141638,C19H22O6,FNEADFUPWHAVTA-PPDYZMKSSA-N,FNEADFUPWHAVTA,"[FNEADFUPWHAVTA-PPDYZMKSSA-N, FNEADFUPWHAVTA-P...","[FNEADFUPWHAVTA, FNEADFUPWHAVTA]",[CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H...,"[FNEADFUPWHAVTA-PPDYZMKSSA-N, FNEADFUPWHAVTA-P...",FNEADFUPWHAVTA
1,BGC0000002,['aculeximycin'],['CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(...,[1672.9651350679992],['C81H144N2O33'],aculeximycin,CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C)...,1672.965135,C81H144N2O33,VJKZKLDZOAFAEE-QIESNYARSA-N,VJKZKLDZOAFAEE,[VJKZKLDZOAFAEE-QIESNYARSA-N],[VJKZKLDZOAFAEE],[CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C...,VJKZKLDZOAFAEE-QIESNYARSA-N,VJKZKLDZOAFAEE


In [89]:
# Save to CSV (without columns: main_compound, main_structure, main_mass, main_formula, InChIKey_smiles, inchikey_connectivity and structures_list)
mibig_df01 = mibig_df.drop(columns=['main_compound', 'main_structure', 'main_mass', 'main_formula', 'InChIKey_smiles', 'inchikey_connectivity', 'structures_list'])
mibig_df01.to_csv('/home/fab25/datasets/mibig/mibig_compounds.csv', index=False)

- Method 2

In [14]:
# Predicting InchiKey for all available SMILES per BGC
from ast import literal_eval

# 1. Turn the 'structures' string into a real list 
mibig_df['structures_list'] = mibig_df['structures'].apply(literal_eval)

# 2. Explode so each SMILES is its own row
exploded = mibig_df[['accession', 'structures_list']].explode('structures_list')
exploded = exploded.rename(columns={'structures_list': 'smiles'})

# 3. Define the converter (SMILES -> InchiKey)
def smiles_to_inchikey_connectivity(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if not smiles or not isinstance(smiles, str):
            return None, None
        if not mol:
            return None, None
        ik = inchi.MolToInchiKey(mol)
        return ik, ik.split('-')[0]
    except Exception as e:
        print(f"Error on {smiles}: {e}")
        return None, None

# 4. Apply to each SMILES
exploded[['InChIKey_full', 'InChIKey_conn']] = exploded['smiles'] \
    .apply(lambda smi: pd.Series(smiles_to_inchikey_connectivity(smi)))

# 5. Re-aggregate: one row per accession, lists of kyes
result = exploded.groupby('accession').agg({
    'smiles': list,
    'InChIKey_full': list,
    'InChIKey_conn': list
}).reset_index()

# 6. Join back any other columns 
final_df = mibig_df.drop(columns=['structures_list'])\
    .merge(result, on='accession', how='left')

[23:45:47] Conflicting single bond directions around double bond at index 26.
[23:45:47]   BondStereo set to STEREONONE and single bond directions set to NONE.
[23:45:48] WARNING: not removing hydrogen atom without neighbors
[23:45:48] WARNING: not removing hydrogen atom without neighbors
[23:45:48] WARNING: not removing hydrogen atom without neighbors
[23:45:48] WARNING: not removing hydrogen atom without neighbors
[23:45:51] WARNING: not removing hydrogen atom without neighbors


Error on nan: No registered converter was able to produce a C++ rvalue of type std::basic_string<wchar_t, std::char_traits<wchar_t>, std::allocator<wchar_t> > from this Python object of type float
Error on nan: No registered converter was able to produce a C++ rvalue of type std::basic_string<wchar_t, std::char_traits<wchar_t>, std::allocator<wchar_t> > from this Python object of type float
Error on nan: No registered converter was able to produce a C++ rvalue of type std::basic_string<wchar_t, std::char_traits<wchar_t>, std::allocator<wchar_t> > from this Python object of type float
Error on nan: No registered converter was able to produce a C++ rvalue of type std::basic_string<wchar_t, std::char_traits<wchar_t>, std::allocator<wchar_t> > from this Python object of type float
Error on nan: No registered converter was able to produce a C++ rvalue of type std::basic_string<wchar_t, std::char_traits<wchar_t>, std::allocator<wchar_t> > from this Python object of type float
Error on nan: N

In [15]:
final_df.head()

,accession,compound_names,structures,masses,formulas,main_compound,main_structure,main_mass,main_formula,InChIKey_smiles,inchikey_connectivity,smiles,InChIKey_full,InChIKey_conn
0,BGC0000001,"['abyssomicin C', 'atrop-abyssomicin C']",['CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\\C=C/C(=O)[C@...,"[346.141638424, 346.141638424]","['C19H22O6', 'C19H22O6']",abyssomicin C,CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H]...,346.141638,C19H22O6,FNEADFUPWHAVTA-PPDYZMKSSA-N,FNEADFUPWHAVTA,[CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H...,"[FNEADFUPWHAVTA-PPDYZMKSSA-N, FNEADFUPWHAVTA-P...","[FNEADFUPWHAVTA, FNEADFUPWHAVTA]"
1,BGC0000002,['aculeximycin'],['CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(...,[1672.9651350679992],['C81H144N2O33'],aculeximycin,CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C)...,1672.965135,C81H144N2O33,VJKZKLDZOAFAEE-QIESNYARSA-N,VJKZKLDZOAFAEE,[CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C...,[VJKZKLDZOAFAEE-QIESNYARSA-N],[VJKZKLDZOAFAEE]
2,BGC0000003,['AF-toxin'],['CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C...,[440.20463260399987],['C22H32O9'],AF-toxin,CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C)O...,440.204633,C22H32O9,ONOBRFRRMLDPES-BGSVYHRFSA-N,ONOBRFRRMLDPES,[CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C)...,[ONOBRFRRMLDPES-BGSVYHRFSA-N],[ONOBRFRRMLDPES]
3,BGC0000004,['aflatoxin G1'],['COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[...,[328.058302724],['C17H12O7'],aflatoxin G1,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[C@...,328.058303,C17H12O7,XWIYFDMXXLINPU-RBHXEPJQSA-N,XWIYFDMXXLINPU,[COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[C...,[XWIYFDMXXLINPU-RBHXEPJQSA-N],[XWIYFDMXXLINPU]
4,BGC0000005,['aflatoxin'],['COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4...,[328.058302724],['C17H12O7'],aflatoxin,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4=C1,328.058303,C17H12O7,XWIYFDMXXLINPU-UHFFFAOYSA-N,XWIYFDMXXLINPU,[COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4=C1],[XWIYFDMXXLINPU-UHFFFAOYSA-N],[XWIYFDMXXLINPU]


#### 4.2) Load the cleaned gnps file 

In [86]:
cleaned_gnps = pd.read_csv(os.path.join(root_gnps_path, 'ALL_GNPS_cleaned.csv'))
print(f'shape of cleaned gnps df: {cleaned_gnps.shape}')
cleaned_gnps.head()

/tmp/ipykernel_1497877/2804238892.py:1: DtypeWarning: Columns (4,13,16) have mixed types. Specify dtype option on import or set low_memory=False.
  cleaned_gnps = pd.read_csv(os.path.join(root_gnps_path, 'ALL_GNPS_cleaned.csv'))


shape of cleaned gnps df: (542777, 19)


,scan,spectrum_id,collision_energy,Adduct,Compound_Source,Compund_Name,Precursor_MZ,ExactMass,Charge,Ion_Mode,Smiles,INCHI,InChIKey_smiles,msManufacturer,msMassAnalyzer,msIonisation,msDissociationMethod,GNPS_library_membership,ppmBetweenExpAndThMass
0,1,CCMSLIB00000001547,NaN,[M+H]1+,isolated,3-Des-Microcystein_LR,981.540,980.533118,1,positive,CC(C=CC1NC(=O)C(CCCN=C(N)N)NC(=O)C(C)C(C(=O)O)...,InChI=1S/C48H72N10O12/c1-25(2)22-36-45(66)57-3...,UYJGHPVHCMVZPP-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,0.406263
1,2,CCMSLIB00000001548,NaN,[M+H]1+,isolated,Hoiamide B,940.250,939.451957,1,positive,CCCC(C)C(O)C(C)C1OC(=O)C(C(C)O)NC(=O)C(C(C)CC)...,InChI=1S/C45H73N5O10S3/c1-14-17-24(6)34(52)26(...,KNGPFNUOXXLKCN-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,222.483646
2,3,CCMSLIB00000001549,NaN,[M+H]1+,isolated,Malyngamide C,456.100,455.243851,1,positive,CCCCCCCC(CC=CCCC(=O)NCC(=CCl)C12OC1C(O)CCC2=O)OC,InChI=1S/C24H38ClNO5/c1-3-4-5-6-8-11-19(30-2)1...,WXDBUBIFYCCNLE-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,331.235723
3,4,CCMSLIB00000001550,NaN,[M+H]1+,isolated,Scytonemin,545.000,544.142307,1,positive,O=C1C(=Cc2ccc(O)cc2)C2=Nc3ccccc3C2=C1C1=C2C(=N...,InChI=1S/C36H20N2O4/c39-21-13-9-19(10-14-21)17...,CGZKSPLDUIRCIO-UHFFFAOYSA-N,NaN,ion trap,ESI,NaN,GNPS-LIBRARY,274.392751
4,5,CCMSLIB00000001551,NaN,[M+H]1+,isolated,Salinisporamide A,314.116,NaN,1,positive,NaN,NaN,NaN,NaN,qtof,ESI,NaN,GNPS-LIBRARY,NaN


In [90]:
# Add the inchikey_connectivity column to the cleaned GNPS DataFrame
# Only apply to rows where InChIKey_smiles is not null or 'N/A'
cleaned_gnps['inchikey_connectivity'] = cleaned_gnps['InChIKey_smiles'].apply(
    lambda x: x.split('-')[0] if pd.notna(x) and x != 'N/A' else None
)

cleaned_gnps.head()

,scan,spectrum_id,collision_energy,Adduct,Compound_Source,Compund_Name,Precursor_MZ,ExactMass,Charge,Ion_Mode,Smiles,INCHI,InChIKey_smiles,msManufacturer,msMassAnalyzer,msIonisation,msDissociationMethod,GNPS_library_membership,ppmBetweenExpAndThMass,inchikey_connectivity
0,1,CCMSLIB00000001547,NaN,[M+H]1+,isolated,3-Des-Microcystein_LR,981.540,980.533118,1,positive,CC(C=CC1NC(=O)C(CCCN=C(N)N)NC(=O)C(C)C(C(=O)O)...,InChI=1S/C48H72N10O12/c1-25(2)22-36-45(66)57-3...,UYJGHPVHCMVZPP-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,0.406263,UYJGHPVHCMVZPP
1,2,CCMSLIB00000001548,NaN,[M+H]1+,isolated,Hoiamide B,940.250,939.451957,1,positive,CCCC(C)C(O)C(C)C1OC(=O)C(C(C)O)NC(=O)C(C(C)CC)...,InChI=1S/C45H73N5O10S3/c1-14-17-24(6)34(52)26(...,KNGPFNUOXXLKCN-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,222.483646,KNGPFNUOXXLKCN
2,3,CCMSLIB00000001549,NaN,[M+H]1+,isolated,Malyngamide C,456.100,455.243851,1,positive,CCCCCCCC(CC=CCCC(=O)NCC(=CCl)C12OC1C(O)CCC2=O)OC,InChI=1S/C24H38ClNO5/c1-3-4-5-6-8-11-19(30-2)1...,WXDBUBIFYCCNLE-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,331.235723,WXDBUBIFYCCNLE
3,4,CCMSLIB00000001550,NaN,[M+H]1+,isolated,Scytonemin,545.000,544.142307,1,positive,O=C1C(=Cc2ccc(O)cc2)C2=Nc3ccccc3C2=C1C1=C2C(=N...,InChI=1S/C36H20N2O4/c39-21-13-9-19(10-14-21)17...,CGZKSPLDUIRCIO-UHFFFAOYSA-N,NaN,ion trap,ESI,NaN,GNPS-LIBRARY,274.392751,CGZKSPLDUIRCIO
4,5,CCMSLIB00000001551,NaN,[M+H]1+,isolated,Salinisporamide A,314.116,NaN,1,positive,NaN,NaN,NaN,NaN,qtof,ESI,NaN,GNPS-LIBRARY,NaN,None


### 4.3) Add BGC ID from mibig_df file based on the joint inchikey_connectivity

In [91]:
mibig_df.head()

,accession,compound_names,structures,masses,formulas,main_compound,main_structure,main_mass,main_formula,InChIKey_smiles,inchikey_connectivity,inchikeys,connectivity_layers,structures_list,inchikey,connectivity_layer
0,BGC0000001,"['abyssomicin C', 'atrop-abyssomicin C']",['CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\\C=C/C(=O)[C@...,"[346.141638424, 346.141638424]","['C19H22O6', 'C19H22O6']",abyssomicin C,CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H]...,346.141638,C19H22O6,FNEADFUPWHAVTA-PPDYZMKSSA-N,FNEADFUPWHAVTA,"[FNEADFUPWHAVTA-PPDYZMKSSA-N, FNEADFUPWHAVTA-P...","[FNEADFUPWHAVTA, FNEADFUPWHAVTA]",[CC1C[C@]23OC(=O)C4=C2OC1C(O)C3\C=C/C(=O)[C@@H...,"[FNEADFUPWHAVTA-PPDYZMKSSA-N, FNEADFUPWHAVTA-P...",FNEADFUPWHAVTA
1,BGC0000002,['aculeximycin'],['CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(...,[1672.9651350679992],['C81H144N2O33'],aculeximycin,CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C)...,1672.965135,C81H144N2O33,VJKZKLDZOAFAEE-QIESNYARSA-N,VJKZKLDZOAFAEE,[VJKZKLDZOAFAEE-QIESNYARSA-N],[VJKZKLDZOAFAEE],[CCCC(O[C@H]1C[C@](C)(N)[C@H](O)[C@H](C)O1)C(C...,VJKZKLDZOAFAEE-QIESNYARSA-N,VJKZKLDZOAFAEE
2,BGC0000003,['AF-toxin'],['CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C...,[440.20463260399987],['C22H32O9'],AF-toxin,CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C)O...,440.204633,C22H32O9,ONOBRFRRMLDPES-BGSVYHRFSA-N,ONOBRFRRMLDPES,[ONOBRFRRMLDPES-BGSVYHRFSA-N],[ONOBRFRRMLDPES],[CCC(C)C(C(=O)OC(/C=C/C=C/C=C/C(=O)O)C1(CO1)C)...,ONOBRFRRMLDPES-BGSVYHRFSA-N,ONOBRFRRMLDPES
3,BGC0000004,['aflatoxin G1'],['COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[...,[328.058302724],['C17H12O7'],aflatoxin G1,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[C@...,328.058303,C17H12O7,XWIYFDMXXLINPU-RBHXEPJQSA-N,XWIYFDMXXLINPU,[XWIYFDMXXLINPU-RBHXEPJQSA-N],[XWIYFDMXXLINPU],[COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4[C@H]5C=CO[C...,XWIYFDMXXLINPU-RBHXEPJQSA-N,XWIYFDMXXLINPU
4,BGC0000005,['aflatoxin'],['COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4...,[328.058302724],['C17H12O7'],aflatoxin,COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4=C1,328.058303,C17H12O7,XWIYFDMXXLINPU-UHFFFAOYSA-N,XWIYFDMXXLINPU,[XWIYFDMXXLINPU-UHFFFAOYSA-N],[XWIYFDMXXLINPU],[COC1=C2C3=C(C(=O)OCC3)C(=O)OC2=C4C5C=COC5OC4=C1],XWIYFDMXXLINPU-UHFFFAOYSA-N,XWIYFDMXXLINPU


In [101]:
# Ensure both DataFrames have the column and it's clean
mibig_df = mibig_df.dropna(subset=['connectivity_layer'])
cleaned_gnps1 = cleaned_gnps.dropna(subset=['inchikey_connectivity'])

# Explode connectivity_layer so each value is a string not a list
mibig_exploded = mibig_df.explode('connectivity_layer')

# Set up the mapping from inchikey_connectivity to accession
# Drop duplicates to ensure unique index
connectivity_to_bgc = (
    mibig_exploded.drop_duplicates(subset='connectivity_layer')
            .set_index('connectivity_layer')['accession']
)
# Map BGC accession onto cleaned_gnps based on shared connectivity
cleaned_gnps1['BGC_accession'] = cleaned_gnps1['inchikey_connectivity'].map(connectivity_to_bgc)

/tmp/ipykernel_1497877/1516056095.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_gnps1['BGC_accession'] = cleaned_gnps1['inchikey_connectivity'].map(connectivity_to_bgc)


In [19]:
# Map all available InChIKeys per BGC to spectra 
# 1. Explode the final_df so every InChIKey_conn is its own row
bgc_conn_map = (
    final_df[['accession', 'InChIKey_conn']]
      .explode('InChIKey_conn')                    # one key per row
      .dropna(subset=['InChIKey_conn'])            # drop empty keys
)

# 2. Group by connectivity key → list of accessions, as couple of BGCs might have the same connectivity key
conn_to_accessions = (
    bgc_conn_map
      .groupby('InChIKey_conn')['accession']
      .agg(list)                                   # collect into Python lists
)

# 3. Map each spectrum’s single key onto the list of BGCs
cleaned_gnps1['BGC_accessions'] = (
    cleaned_gnps1['inchikey_connectivity']
              .map(conn_to_accessions)             # series of lists or NaN
              .apply(lambda x: x if isinstance(x, list) else [])  
              # replace NaN with empty list
)

# 4. Count how many spectra got ≥1 BGC hit
n_with_hits = (cleaned_gnps1['BGC_accessions']
                   .apply(bool)                    # True if list non‑empty
                   .sum())
print(f"{n_with_hits} spectra out of {len(cleaned_gnps1)} have ≥1 BGC accession")


9322 spectra out of 487394 have ≥1 BGC accession


/tmp/ipykernel_4184526/1161248729.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_gnps1['BGC_accessions'] = (


In [20]:
# count via the new list‑column
n_with_hits_new = cleaned_gnps1['BGC_accessions'].apply(bool).sum()
print(f"{n_with_hits_new} spectra out of {len(cleaned_gnps1)} have ≥1 BGC accession (new way)")


9322 spectra out of 487394 have ≥1 BGC accession (new way)


In [21]:
cleaned_gnps1.head(5)

,scan,spectrum_id,collision_energy,Adduct,Compound_Source,Compund_Name,Precursor_MZ,ExactMass,Charge,Ion_Mode,...,InChIKey_smiles,msManufacturer,msMassAnalyzer,msIonisation,msDissociationMethod,GNPS_library_membership,ppmBetweenExpAndThMass,inchikey_connectivity,BGC_accession,BGC_accessions
0,1,CCMSLIB00000001547,NaN,[M+H]1+,isolated,3-Des-Microcystein_LR,981.540,980.533118,1,positive,...,UYJGHPVHCMVZPP-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,0.406263,UYJGHPVHCMVZPP,NaN,[]
1,2,CCMSLIB00000001548,NaN,[M+H]1+,isolated,Hoiamide B,940.250,939.451957,1,positive,...,KNGPFNUOXXLKCN-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,222.483646,KNGPFNUOXXLKCN,NaN,[]
2,3,CCMSLIB00000001549,NaN,[M+H]1+,isolated,Malyngamide C,456.100,455.243851,1,positive,...,WXDBUBIFYCCNLE-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,331.235723,WXDBUBIFYCCNLE,NaN,[]
3,4,CCMSLIB00000001550,NaN,[M+H]1+,isolated,Scytonemin,545.000,544.142307,1,positive,...,CGZKSPLDUIRCIO-UHFFFAOYSA-N,NaN,ion trap,ESI,NaN,GNPS-LIBRARY,274.392751,CGZKSPLDUIRCIO,NaN,[]
5,6,CCMSLIB00000001552,NaN,[M+H]1+,isolated,Hectochlorin,667.115,666.105328,1,positive,...,USXIYWCPCGVOKF-CRTVXBCISA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,3006.165539,USXIYWCPCGVOKF,BGC0001000,[BGC0001000]


In [22]:
cleaned_gnps1.BGC_accession.isna().value_counts()

BGC_accession
True     479461
False      7933
Name: count, dtype: int64

In [23]:
# Check how many new spectra were assigned to some BGCs 
old_unmatched = cleaned_gnps1['BGC_accession'].isna()
new_matched  = cleaned_gnps1['BGC_accessions'].apply(bool)

switched = cleaned_gnps1[old_unmatched & new_matched]
print(f"{len(switched)} spectra were newly matched under the one‑to‑many scheme")


1389 spectra were newly matched under the one‑to‑many scheme


In [24]:
# keep only rows where the list has length >= 1
matched_spectra = cleaned_gnps1[cleaned_gnps1['BGC_accessions'].apply(len) >= 1]

print(f"{len(matched_spectra)} spectra with >=1 BGC accession")


9322 spectra with >=1 BGC accession


### 4.4) Save & load BGC matched spectra

In [104]:
root_gnps_path

'/home/fab25/datasets/gnps'

In [105]:
# ### Save the updated cleaned GNPS DataFrame
# matched_spectra.to_csv(os.path.join(root_gnps_path, 'gnps_mibig_match01.csv'), index=False)

root_match_path = '/home/fab25/datasets/gnps_mibig_match'
### Load BGC matched spectra 
matched_spectra = pd.read_csv(os.path.join(root_match_path, 'gnps_mibig_match01.csv'))
print(f'Shape of the BGC matched spectra: {matched_spectra.shape}')
matched_spectra.head()

Shape of the BGC matched spectra: (9322, 22)


,scan,spectrum_id,collision_energy,Adduct,Compound_Source,Compund_Name,Precursor_MZ,ExactMass,Charge,Ion_Mode,...,InChIKey_smiles,msManufacturer,msMassAnalyzer,msIonisation,msDissociationMethod,GNPS_library_membership,ppmBetweenExpAndThMass,inchikey_connectivity,BGC_accession,BGC_accessions
0,6,CCMSLIB00000001552,NaN,[M+H]1+,isolated,Hectochlorin,667.115,666.105328,1,positive,...,USXIYWCPCGVOKF-CRTVXBCISA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,3006.165539,USXIYWCPCGVOKF,BGC0001000,['BGC0001000']
1,7,CCMSLIB00000001553,NaN,[M+H]1+,isolated,Hectochlorin,689.000,664.108278,1,positive,...,USXIYWCPCGVOKF-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,35910.222460,USXIYWCPCGVOKF,BGC0001000,['BGC0001000']
2,9,CCMSLIB00000001555,NaN,[M+H]1+,isolated,Hectochlorin,665.115,664.108278,1,positive,...,USXIYWCPCGVOKF-UHFFFAOYSA-N,NaN,ftms,ESI,NaN,GNPS-LIBRARY,0.830753,USXIYWCPCGVOKF,BGC0001000,['BGC0001000']
3,12,CCMSLIB00000001558,NaN,[M+Na]1+,isolated,Hectochlorin,687.000,664.108278,1,positive,...,USXIYWCPCGVOKF-UHFFFAOYSA-N,NaN,ftms,ESI,NaN,GNPS-LIBRARY,141.896594,USXIYWCPCGVOKF,BGC0001000,['BGC0001000']
4,14,CCMSLIB00000001560,NaN,[M+H]1+,isolated,Jamaicamide A,569.000,566.154697,1,positive,...,NAIKIJSSBJHCBL-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,3240.739809,NAIKIJSSBJHCBL,BGC0001001,['BGC0001001']


In [106]:
### Keep only unique BGCs (keep the first occurrence)
unique_bgcs = cleaned_gnps1[~cleaned_gnps1['BGC_accession'].isna()].drop_duplicates(subset='BGC_accession', keep='first')
unique_bgcs

,scan,spectrum_id,collision_energy,Adduct,Compound_Source,Compund_Name,Precursor_MZ,ExactMass,Charge,Ion_Mode,...,INCHI,InChIKey_smiles,msManufacturer,msMassAnalyzer,msIonisation,msDissociationMethod,GNPS_library_membership,ppmBetweenExpAndThMass,inchikey_connectivity,BGC_accession
5,6,CCMSLIB00000001552,NaN,[M+H]1+,isolated,Hectochlorin,667.1150,666.105328,1,positive,...,"InChI=1S/C27H34Cl2N2O9S2/c1-13-17(9-8-10-27(7,...",USXIYWCPCGVOKF-CRTVXBCISA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,3006.165539,USXIYWCPCGVOKF,BGC0001000
13,14,CCMSLIB00000001560,NaN,[M+H]1+,isolated,Jamaicamide A,569.0000,566.154697,1,positive,...,InChI=1S/C27H36BrClN2O4/c1-21(12-14-23(20-29)1...,NAIKIJSSBJHCBL-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,3240.739809,NAIKIJSSBJHCBL,BGC0001001
59,60,CCMSLIB00000001606,NaN,[M+H]1+,isolated,Emericellamide A,610.4160,609.410149,1,positive,...,InChI=1S/C31H55N5O7/c1-10-11-12-13-14-19(6)26-...,QURRTAYEASAREY-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,2.339739,QURRTAYEASAREY,BGC0001290
66,67,CCMSLIB00000001613,NaN,[M+H]1+,isolated,Cyclomarin D,995.5980,1012.599741,1,positive,...,"InChI=1S/C55H80N8O10/c1-15-55(11,12)63-28-38(3...",AHDUXXXZGSWYHF-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,17767.261519,AHDUXXXZGSWYHF,BGC0000333
68,69,CCMSLIB00000001615,NaN,[M+H]1+,isolated,Pacificanone A,323.2580,322.250795,1,positive,...,"InChI=1S/C20H34O3/c1-7-17-12-15(5)20(23,16(6)1...",JSPPQWVTDRBUIB-UHFFFAOYSA-N,NaN,qtof,ESI,NaN,GNPS-LIBRARY,0.224414,JSPPQWVTDRBUIB,BGC0001830
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498262,498263,MSBNK-AAFC-AC000655,20.0,[M-H]1-,NaN,Fumonisin B4,688.3919,689.398641,-1,negative,...,InChI=1S/C34H59NO13/c1-5-6-14-22(3)32(48-31(42...,WYYKRDVIBOEORL-UHFFFAOYSA-N,Thermo,ftms,ESI,hcd,MassBank_ML_Export,0.770722,WYYKRDVIBOEORL,BGC0002904
501099,501100,MSBNK-Keio_Univ-KO002711,10.0,[M+H]1+,NaN,2-Deoxystreptamine,163.0000,162.100442,1,positive,...,InChI=1S/C6H14N2O3/c7-2-1-3(8)5(10)6(11)4(2)9/...,DTFAJAKTSMLKAT-UHFFFAOYSA-N,NaN,quadrupole,NaN,NaN,MassBank_ML_Export,660.421846,DTFAJAKTSMLKAT,BGC0000689
521067,521068,MSBNK-AAFC-AC000095,10.0,[M+H]1+,NaN,Culmorin,239.2000,238.193280,1,positive,...,InChI=1S/C15H26O2/c1-13(2)6-5-7-14(3)10-9(16)8...,VWMGBHVRRNKOAE-UHFFFAOYSA-N,Thermo,ftms,ESI,hcd,MassBank_ML_Export,2.329788,VWMGBHVRRNKOAE,BGC0002176
530611,530612,MSBNK-Washington_State_Univ-BML00097,10.0,[M+H]1+,NaN,Loline,155.1179,154.110613,1,positive,...,InChI=1S/C8H14N2O/c1-9-7-6-4-10-3-2-5(11-6)8(7...,OPMNROCQHKJDAQ-UHFFFAOYSA-N,Agilent,qtof,NaN,NaN,MassBank_ML_Export,0.065102,OPMNROCQHKJDAQ,BGC0000815


In [ ]:
# # Save unique BGC to spectrum link 
# unique_bgcs.to_csv(os.path.join(root_gnps_path, 'gnps_mibig_unique_bgcs.csv'), index=False)

### 4.5) Predict InChIKey connectivity layer for spectra from original json 

In [107]:
gnps_df

,spectrum_id,Compound_Name,Precursor_MZ,ExactMass,Smiles,INCHI,InChIKey_smiles
0,CCMSLIB00000001547,3-Des-Microcystein_LR,981.540,0.000,CC(C)CC1NC(=O)C(C)NC(=O)C(=C)N(C)C(=O)CCC(NC(=...,NaN,NaN
1,CCMSLIB00000001548,Hoiamide B,940.250,939.452,CCC[C@@H](C)[C@@H]([C@H](C)[C@@H]1[C@H]([C@H](...,InChI=1S/C45H73N5O10S3/c1-14-17-24(6)34(52)26(...,NaN
2,CCMSLIB00000001549,Malyngamide C,456.100,455.244,CCCCCCC[C@@H](C/C=C/CCC(=O)NC/C(=C/Cl)/[C@@]12...,InChI=1S/C24H38ClNO5/c1-3-4-5-6-8-11-19(30-2)1...,NaN
3,CCMSLIB00000001550,Scytonemin,545.000,0.000,OC1=CC=C(\C=C2\C(=O)C(C3=C4C5=C(C=CC=C5)N=C4\C...,InChI=1S/C36H20N2O4/c39-21-13-9-19(10-14-21)17...,NaN
4,CCMSLIB00000001551,Salinisporamide A,314.116,313.108,NaN,NaN,NaN
...,...,...,...,...,...,...,...
1299129,CCMSLIB00006126839,1773416 - 40.0 eV,172.098,0.000,CCCCCC(=O)NCC(=O)O,NaN,NaN
1299130,CCMSLIB00006126840,1773416 - 40.0 eV,172.098,0.000,CCCCCC(=O)NCC(=O)O,NaN,NaN
1299131,CCMSLIB00006126841,1773416 - 30.0 eV,172.098,0.000,CCCCCC(=O)NCC(=O)O,NaN,NaN
1299132,CCMSLIB00006126842,1773416 - 30.0 eV,172.098,0.000,CCCCCC(=O)NCC(=O)O,NaN,NaN


In [108]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import inchi
from tqdm import tqdm

tqdm.pandas()

def compute_inchikey_pair(smiles: str) -> pd.Series:
    """
    input: 
        - smiles: SMILES structure
    output:
        - InchiKey, generated based on the SMILES structure 
    """
    # … same as before …
    if not isinstance(smiles, str):
        return pd.Series([None, None])
    s = smiles.strip().lower()
    if not s or s in {'n/a', 'nan'} or s.startswith('inchi='):
        return pd.Series([None, None])
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            key = inchi.MolToInchiKey(mol)
            return pd.Series([key, key.split('-')[0]])
    except:
        pass
    return pd.Series([None, None])

In [109]:
# 1) Take only the first x rows
small_df = gnps_df.iloc[:].copy()

# 2) Build the mask on that slice
# Ensure 'Smiles' is a string (to avoid errors). Replaces NaN values with an empty string ''
smiles_series = small_df['Smiles'].fillna('').astype(str)

# Now apply the mask safely
mask_small = (
    small_df['InChIKey_smiles'].isna()                      # select spectra without InChIKey (whole file in this case)
    & (smiles_series.str.strip() != '')
    & (~smiles_series.str.lower().str.startswith('inchi='))
)


In [38]:
# 3) Apply your function (this RETURNS a Series of pd.Series!)
results = small_df.loc[mask_small, 'Smiles'] \
                  .progress_apply(compute_inchikey_pair)

# 4) Turn new object, Series-of-Series, into a real DataFrame
results_df = pd.DataFrame(
    results.values.tolist(),                # Convert Series of Series into DataFrame 
    index=results.index,                    # Keep the original index
    columns=['InChIKey_smiles','inchikey_connectivity']
)

# Add two new columns to the original dataframe if they don't already exist 
for col in ['InChIKey_smiles', 'inchikey_connectivity']:
    if col not in small_df.columns:
        small_df[col] = None

# 5) Before/after sanity checks
print("Before fill:", small_df.loc[mask_small, 'InChIKey_smiles'].isna().sum())
small_df.loc[mask_small, ['InChIKey_smiles','inchikey_connectivity']] = results_df
print(" After fill:", small_df.loc[mask_small, 'InChIKey_smiles'].isna().sum())

  0%|          | 0/1079275 [00:00<?, ?it/s]

  0%|          | 1109/1079275 [00:01<17:01, 1055.04it/s][22:15:23] Can't kekulize mol.  Unkekulized atoms: 10 11 12 14 16
[22:15:23] Can't kekulize mol.  Unkekulized atoms: 10 11 12 14 16
  0%|          | 1216/1079275 [00:01<18:26, 974.44it/s] [22:15:23] SMILES Parse Error: unclosed ring for input: 'OC1=CC(C(OC)=O)=C(OC2=CC(C)=CC(O)=C2C(O)=O)C(OC)=C2'
[22:15:23] SMILES Parse Error: unclosed ring for input: 'O=C1C2=C(C=C(C)C=C2O)OC3=CC(O)=CC(C(OC)=O)=C32'
  0%|          | 1316/1079275 [00:01<18:55, 949.43it/s][22:15:23] SMILES Parse Error: unclosed ring for input: 'O=C([C@H](CC)C)O[C@H]1CCC=C2C1[C@@H](CC[C@@H](O)C[C@@H](O)CC(OC)=O)[C@@H](C)C=C3'
[22:15:23] SMILES Parse Error: unclosed ring for input: 'O=C(N[C@@H](CCCCCC(CC)=O)C(N[C@@H](CC1=CN(OC)C2=C1C=CC=C2)C3=O)=O)[C@@H]4N(C([C@H]([C@H](CC)C)N3)=O)CCCC5'
[22:15:23] SMILES Parse Error: unclosed ring for input: 'O=C(N(C(C=CC=C1)=C1C(N(C)[C@@]2([H])CC3=CC=CC=C3)=O)C2=N4)C5=C4C=CC=C6'
[22:15:23] SMILES Parse Error: unclosed ring for input

Before fill: 1079275
 After fill: 982


- The reason that still there is no InchIKey for some of the spectra could be that no SMILES suggested for them. 

In [ ]:
# from rdkit import Chem
# from rdkit.Chem import inchi
# import pandas as pd
# from tqdm import tqdm

# # Register tqdm with pandas `.apply`
# tqdm.pandas()

# def compute_inchikey_pair(smiles: str) -> pd.Series:
#     """
#     Converts a SMILES string to an InChIKey and its connectivity block.
#     Returns (None, None) if conversion fails.
#     """
#     if not isinstance(smiles, str) or not smiles.strip() or smiles.strip().lower() == 'nan':
#         return pd.Series([None, None])

#     try:
#         mol = Chem.MolFromSmiles(smiles)
#         if mol:
#             key = inchi.MolToInchiKey(mol)
#             return pd.Series([key, key.split('-')[0]])
#     except Exception:
#         pass
#     return pd.Series([None, None])

# # Prepare a clean mask for rows where InChIKey is missing but SMILES is valid
# gnps_df['Smiles'] = gnps_df['Smiles'].astype(str).str.strip()
# mask_missing = (
#     gnps_df['InChIKey_smiles'].isna()
#     & gnps_df['Smiles'].notna()
#     & (gnps_df['Smiles'] != '')
#     & (gnps_df['Smiles'].str.lower() != 'nan')
# )

# # Use tqdm to track progress of apply
# gnps_df.loc[mask_missing, ['InChIKey_smiles', 'inchikey_connectivity']] = \
#     gnps_df.loc[mask_missing, 'Smiles'].progress_apply(compute_inchikey_pair)

  0%|          | 921/1079280 [00:01<17:55, 1002.97it/s][18:38:44] SMILES Parse Error: syntax error while parsing: N/A
[18:38:44] SMILES Parse Error: Failed parsing SMILES 'N/A' for input: 'N/A'
[18:38:44] SMILES Parse Error: syntax error while parsing: N/A
[18:38:44] SMILES Parse Error: Failed parsing SMILES 'N/A' for input: 'N/A'
[18:38:44] SMILES Parse Error: syntax error while parsing: InChI=1S/C16H21NO2/c1-2-3-4-5-6-11-14-16(19)15(18)12-9-7-8-10-13(12)17-14/h7-10,19H,2-6,11H2,1H3,(H,17,18)
[18:38:44] SMILES Parse Error: Failed parsing SMILES 'InChI=1S/C16H21NO2/c1-2-3-4-5-6-11-14-16(19)15(18)12-9-7-8-10-13(12)17-14/h7-10,19H,2-6,11H2,1H3,(H,17,18)' for input: 'InChI=1S/C16H21NO2/c1-2-3-4-5-6-11-14-16(19)15(18)12-9-7-8-10-13(12)17-14/h7-10,19H,2-6,11H2,1H3,(H,17,18)'
  0%|          | 1141/1079280 [00:01<17:43, 1013.64it/s][18:38:44] Can't kekulize mol.  Unkekulized atoms: 10 11 12 14 16
[18:38:44] Can't kekulize mol.  Unkekulized atoms: 10 11 12 14 16
  0%|          | 1244/1079280 [

In [32]:
# # Save and load modified gnps file 
# small_df.to_csv(os.path.join(root_gnps_path, 'gnps_spectra_with_inchikey.csv'), index=False)

# Load modified gnps spectra 
small_df = pd.read_csv(os.path.join(root_gnps_path, 'gnps_spectra_with_inchikey.csv'))

In [33]:
small_df.head()

,spectrum_id,Compound_Name,Precursor_MZ,ExactMass,Smiles,INCHI,InChIKey_smiles,inchikey_connectivity
0,CCMSLIB00000001547,3-Des-Microcystein_LR,981.540,0.000,CC(C)CC1NC(=O)C(C)NC(=O)C(=C)N(C)C(=O)CCC(NC(=...,NaN,IYDKWWDUBYWQGF-NNAZGLEUSA-N,IYDKWWDUBYWQGF
1,CCMSLIB00000001548,Hoiamide B,940.250,939.452,CCC[C@@H](C)[C@@H]([C@H](C)[C@@H]1[C@H]([C@H](...,InChI=1S/C45H73N5O10S3/c1-14-17-24(6)34(52)26(...,KNGPFNUOXXLKCN-ZNCJFREWSA-N,KNGPFNUOXXLKCN
2,CCMSLIB00000001549,Malyngamide C,456.100,455.244,CCCCCCC[C@@H](C/C=C/CCC(=O)NC/C(=C/Cl)/[C@@]12...,InChI=1S/C24H38ClNO5/c1-3-4-5-6-8-11-19(30-2)1...,WXDBUBIFYCCNLE-NSCMQRKRSA-N,WXDBUBIFYCCNLE
3,CCMSLIB00000001550,Scytonemin,545.000,0.000,OC1=CC=C(\C=C2\C(=O)C(C3=C4C5=C(C=CC=C5)N=C4\C...,InChI=1S/C36H20N2O4/c39-21-13-9-19(10-14-21)17...,CGZKSPLDUIRCIO-RPCRKUJJSA-N,CGZKSPLDUIRCIO
4,CCMSLIB00000001551,Salinisporamide A,314.116,313.108,NaN,NaN,NaN,NaN


In [34]:
# Ensure both DataFrames have the column and it's clean
mibig_df = mibig_df.dropna(subset=['inchikey_connectivity'])
small_df1 = small_df.dropna(subset=['inchikey_connectivity'])

# Set up the mapping from inchikey_connectivity to accession
# Drop duplicates to ensure unique index
connectivity_to_bgc = (
    mibig_df.drop_duplicates(subset='inchikey_connectivity')
            .set_index('inchikey_connectivity')['accession']
)
# Map BGC accession onto small_df1 based on shared connectivity
small_df1['BGC_accession'] = small_df1['inchikey_connectivity'].map(connectivity_to_bgc)

/tmp/ipykernel_4184526/914311731.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  small_df1['BGC_accession'] = small_df1['inchikey_connectivity'].map(connectivity_to_bgc)


In [35]:
# Map all available InChIKeys per BGC to spectra from original json 
# 1. Explode the final_df so every InChIKey_conn is its own row
bgc_conn_map = (
    final_df[['accession', 'InChIKey_conn']]
      .explode('InChIKey_conn')                    # one key per row
      .dropna(subset=['InChIKey_conn'])            # drop empty keys
)

# 2. Group by connectivity key → list of accessions, as couple of BGCs might have the same connectivity key
conn_to_accessions = (
    bgc_conn_map
      .groupby('InChIKey_conn')['accession']
      .agg(list)                                   # collect into Python lists
)

# 3. Map each spectrum’s single key onto the list of BGCs
small_df1['BGC_accessions'] = (
    small_df1['inchikey_connectivity']
              .map(conn_to_accessions)             # series of lists or NaN
              .apply(lambda x: x if isinstance(x, list) else [])  
              # replace NaN with empty list
)

# 4. Count how many spectra got ≥1 BGC hit
n_with_hits = (small_df1['BGC_accessions']
                   .apply(bool)                    # True if list non‑empty
                   .sum())
print(f"{n_with_hits} spectra out of {len(small_df1)} have ≥1 BGC accession")

12693 spectra out of 1078279 have ≥1 BGC accession


/tmp/ipykernel_4184526/3045039.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  small_df1['BGC_accessions'] = (


In [36]:
small_df1.head()

,spectrum_id,Compound_Name,Precursor_MZ,ExactMass,Smiles,INCHI,InChIKey_smiles,inchikey_connectivity,BGC_accession,BGC_accessions
0,CCMSLIB00000001547,3-Des-Microcystein_LR,981.540,0.000,CC(C)CC1NC(=O)C(C)NC(=O)C(=C)N(C)C(=O)CCC(NC(=...,NaN,IYDKWWDUBYWQGF-NNAZGLEUSA-N,IYDKWWDUBYWQGF,NaN,[]
1,CCMSLIB00000001548,Hoiamide B,940.250,939.452,CCC[C@@H](C)[C@@H]([C@H](C)[C@@H]1[C@H]([C@H](...,InChI=1S/C45H73N5O10S3/c1-14-17-24(6)34(52)26(...,KNGPFNUOXXLKCN-ZNCJFREWSA-N,KNGPFNUOXXLKCN,NaN,[]
2,CCMSLIB00000001549,Malyngamide C,456.100,455.244,CCCCCCC[C@@H](C/C=C/CCC(=O)NC/C(=C/Cl)/[C@@]12...,InChI=1S/C24H38ClNO5/c1-3-4-5-6-8-11-19(30-2)1...,WXDBUBIFYCCNLE-NSCMQRKRSA-N,WXDBUBIFYCCNLE,NaN,[]
3,CCMSLIB00000001550,Scytonemin,545.000,0.000,OC1=CC=C(\C=C2\C(=O)C(C3=C4C5=C(C=CC=C5)N=C4\C...,InChI=1S/C36H20N2O4/c39-21-13-9-19(10-14-21)17...,CGZKSPLDUIRCIO-RPCRKUJJSA-N,CGZKSPLDUIRCIO,NaN,[]
5,CCMSLIB00000001552,Hectochlorin,667.115,0.000,C[C@H]1[C@@H](OC(C2=CSC([C@H](C(C)(OC(C3=CSC([...,,USXIYWCPCGVOKF-LERJCCFDSA-N,USXIYWCPCGVOKF,BGC0001000,[BGC0001000]


In [37]:
# Check how many new spectra were assigned to some BGCs 
old_unmatched = small_df1['BGC_accession'].isna()
new_matched  = small_df1['BGC_accessions'].apply(bool)

switched = small_df1[old_unmatched & new_matched]
print(f"{len(switched)} spectra were newly matched under the one‑to‑many scheme")

1979 spectra were newly matched under the one‑to‑many scheme


In [38]:
# keep only rows where the list has length >= 1
matched_spectra = small_df1[small_df1['BGC_accessions'].apply(len) >= 1]

print(f"{len(matched_spectra)} spectra with >=1 BGC accession")


12693 spectra with >=1 BGC accession


### 4.6) Save and load BGC matched spectra (based on original json file)

In [42]:
# ### Save the updated cleaned GNPS DataFrame
# matched_spectra.to_csv(os.path.join(root_gnps_path, 'gnps_mibig_match_from_original_json01.csv'), index=False)

### Load BGC matched spectra 
matched_spectra = pd.read_csv(os.path.join(root_gnps_path, 'gnps_mibig_match_from_original_json01.csv'))
print(f'Shape of the BGC matched spectra: {matched_spectra.shape}')
matched_spectra.head()

Shape of the BGC matched spectra: (12693, 10)


,spectrum_id,Compound_Name,Precursor_MZ,ExactMass,Smiles,INCHI,InChIKey_smiles,inchikey_connectivity,BGC_accession,BGC_accessions
0,CCMSLIB00000001552,Hectochlorin,667.115,0.000,C[C@H]1[C@@H](OC(C2=CSC([C@H](C(C)(OC(C3=CSC([...,,USXIYWCPCGVOKF-LERJCCFDSA-N,USXIYWCPCGVOKF,BGC0001000,['BGC0001000']
1,CCMSLIB00000001553,Hectochlorin,689.000,664.108,C[C@H]1[C@@H](OC(=O)C2=CSC(=N2)[C@H](C(OC(=O)C...,"InChI=1S/C27H34Cl2N2O9S2/c1-13-17(9-8-10-27(7,...",USXIYWCPCGVOKF-NOENWEJRSA-N,USXIYWCPCGVOKF,BGC0001000,['BGC0001000']
2,CCMSLIB00000001555,Hectochlorin,665.115,0.000,C[C@H]1[C@H](CCCC(C)(Cl)Cl)OC(=O)C2=CSC(=N2)[C...,"InChI=1S/C27H34Cl2N2O9S2/c1-13-17(9-8-10-27(7,...",USXIYWCPCGVOKF-NOENWEJRSA-N,USXIYWCPCGVOKF,BGC0001000,['BGC0001000']
3,CCMSLIB00000001558,Hectochlorin,687.000,0.000,C[C@H]1[C@H](CCCC(C)(Cl)Cl)OC(=O)C2=CSC(=N2)[C...,"InChI=1S/C27H34Cl2N2O9S2/c1-13-17(9-8-10-27(7,...",USXIYWCPCGVOKF-NOENWEJRSA-N,USXIYWCPCGVOKF,BGC0001000,['BGC0001000']
4,CCMSLIB00000001560,Jamaicamide A,569.000,566.155,C[C@H]1C=CC(=O)N1C(=O)/C=C(\CCNC(=O)CC/C=C/C(C...,InChI=1S/C27H36BrClN2O4/c1-21(12-14-23(20-29)1...,NAIKIJSSBJHCBL-VIPNTUGYSA-N,NAIKIJSSBJHCBL,BGC0001001,['BGC0001001']


In [43]:
small_df1[~small_df1.BGC_accession.isna()]

,spectrum_id,Compound_Name,Precursor_MZ,ExactMass,Smiles,INCHI,InChIKey_smiles,inchikey_connectivity,BGC_accession,BGC_accessions
5,CCMSLIB00000001552,Hectochlorin,667.115,0.000,C[C@H]1[C@@H](OC(C2=CSC([C@H](C(C)(OC(C3=CSC([...,,USXIYWCPCGVOKF-LERJCCFDSA-N,USXIYWCPCGVOKF,BGC0001000,[BGC0001000]
6,CCMSLIB00000001553,Hectochlorin,689.000,664.108,C[C@H]1[C@@H](OC(=O)C2=CSC(=N2)[C@H](C(OC(=O)C...,"InChI=1S/C27H34Cl2N2O9S2/c1-13-17(9-8-10-27(7,...",USXIYWCPCGVOKF-NOENWEJRSA-N,USXIYWCPCGVOKF,BGC0001000,[BGC0001000]
8,CCMSLIB00000001555,Hectochlorin,665.115,0.000,C[C@H]1[C@H](CCCC(C)(Cl)Cl)OC(=O)C2=CSC(=N2)[C...,"InChI=1S/C27H34Cl2N2O9S2/c1-13-17(9-8-10-27(7,...",USXIYWCPCGVOKF-NOENWEJRSA-N,USXIYWCPCGVOKF,BGC0001000,[BGC0001000]
11,CCMSLIB00000001558,Hectochlorin,687.000,0.000,C[C@H]1[C@H](CCCC(C)(Cl)Cl)OC(=O)C2=CSC(=N2)[C...,"InChI=1S/C27H34Cl2N2O9S2/c1-13-17(9-8-10-27(7,...",USXIYWCPCGVOKF-NOENWEJRSA-N,USXIYWCPCGVOKF,BGC0001000,[BGC0001000]
13,CCMSLIB00000001560,Jamaicamide A,569.000,566.155,C[C@H]1C=CC(=O)N1C(=O)/C=C(\CCNC(=O)CC/C=C/C(C...,InChI=1S/C27H36BrClN2O4/c1-21(12-14-23(20-29)1...,NAIKIJSSBJHCBL-VIPNTUGYSA-N,NAIKIJSSBJHCBL,BGC0001001,[BGC0001001]
...,...,...,...,...,...,...,...,...,...,...
1299038,CCMSLIB00006126748,(-)-Erythromycin - 40.0 eV,792.475,0.000,CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,NaN,ULGZDMOVFRHVEP-RWJQBGPGSA-N,ULGZDMOVFRHVEP,BGC0000054,"[BGC0000054, BGC0000055]"
1299039,CCMSLIB00006126749,(-)-Erythromycin - 40.0 eV,792.474,0.000,CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,NaN,ULGZDMOVFRHVEP-RWJQBGPGSA-N,ULGZDMOVFRHVEP,BGC0000054,"[BGC0000054, BGC0000055]"
1299040,CCMSLIB00006126750,(-)-Erythromycin - 40.0 eV,792.475,0.000,CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,NaN,ULGZDMOVFRHVEP-RWJQBGPGSA-N,ULGZDMOVFRHVEP,BGC0000054,"[BGC0000054, BGC0000055]"
1299041,CCMSLIB00006126751,(-)-Erythromycin - 40.0 eV,792.474,0.000,CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,NaN,ULGZDMOVFRHVEP-RWJQBGPGSA-N,ULGZDMOVFRHVEP,BGC0000054,"[BGC0000054, BGC0000055]"


In [ ]:
# ### Save the updated cleaned GNPS DataFrame
# small_df1[~small_df1.BGC_accession.isna()].to_csv(os.path.join(root_gnps_path, 'gnps_mibig_match_from_original_json.csv'), index=False)

In [44]:
# Load saved small_df1 
small_df1 = pd.read_csv(os.path.join(root_gnps_path, 'gnps_mibig_match_from_original_json.csv'))
print(f'shape of small_df1: {small_df1.shape}')

shape of small_df1: (10714, 9)


In [45]:
### Keep only unique BGCs (keep the first occurrence)
unique_bgcs1 = small_df1[~small_df1['BGC_accession'].isna()].drop_duplicates(subset='BGC_accession', keep='first')
unique_bgcs1

,spectrum_id,Compound_Name,Precursor_MZ,ExactMass,Smiles,INCHI,InChIKey_smiles,inchikey_connectivity,BGC_accession
0,CCMSLIB00000001552,Hectochlorin,667.1150,0.0000,C[C@H]1[C@@H](OC(C2=CSC([C@H](C(C)(OC(C3=CSC([...,,USXIYWCPCGVOKF-LERJCCFDSA-N,USXIYWCPCGVOKF,BGC0001000
4,CCMSLIB00000001560,Jamaicamide A,569.0000,566.1550,C[C@H]1C=CC(=O)N1C(=O)/C=C(\CCNC(=O)CC/C=C/C(C...,InChI=1S/C27H36BrClN2O4/c1-21(12-14-23(20-29)1...,NAIKIJSSBJHCBL-VIPNTUGYSA-N,NAIKIJSSBJHCBL,BGC0001001
8,CCMSLIB00000001601,Apratoxin A,840.0000,0.0000,CC[C@H](C)[C@H]1N(C)C(=O)[C@H](C)N(C)C(=O)[C@H...,InChI=1S/C45H69N5O8S/c1-13-27(3)38-43(55)50-20...,KXUJXPZXILTXDA-JHMIWIOGSA-N,KXUJXPZXILTXDA,BGC0002542
10,CCMSLIB00000001606,Emericellamide A,610.4160,609.4080,CCCCCC[C@H](C)[C@H]1OC(=O)[C@H](C)NC(=O)[C@H](...,InChI=1S/C31H55N5O7/c1-10-11-12-13-14-19(6)26-...,QURRTAYEASAREY-OOVPVTRWSA-N,QURRTAYEASAREY,BGC0001290
12,CCMSLIB00000001613,Cyclomarin D,995.5980,1012.6000,C[C@H]1C(=O)N[C@H](C(=O)N[C@H](C(=O)N([C@H](C(...,"InChI=1S/C55H80N8O10/c1-15-55(11,12)63-28-38(3...",AHDUXXXZGSWYHF-IXGGKXOYSA-N,AHDUXXXZGSWYHF,BGC0000333
...,...,...,...,...,...,...,...,...,...
8772,CCMSLIB00005727142,"Massbank:AC000176 Kotanin|8-(4,7-dimethoxy-5-m...",439.1380,0.0000,CC1=CC(=C(C2=C1C(=CC(=O)O2)OC)C3=C(C=C(C4=C3OC...,1S/C24H22O8/c1-11-7-13(27-3)21(23-19(11)15(29-...,CSJOUDOXDHMIAH-UHFFFAOYSA-N,CSJOUDOXDHMIAH,BGC0003126
8799,CCMSLIB00005727245,Massbank:AC000543 Cercosporin,535.1590,0.0000,C[C@H](CC1=C(C(=C2C(=O)C=C3C4=C5C(=CC(=O)C6=C(...,1S/C29H26O10/c1-10(30)5-12-18-19-13(6-11(2)31)...,MXLWQNCWIIZUQT-GHMZBOCLSA-N,MXLWQNCWIIZUQT,BGC0001542
8859,CCMSLIB00005727465,Massbank:AC000003 Mellein|Ochracin|8-hydroxy-3...,179.0700,0.0000,CC1CC2=C(C(=CC=C2)O)C(=O)O1,1S/C10H10O3/c1-6-5-7-3-2-4-8(11)9(7)10(12)13-6...,KWILGNNWGSNMPA-UHFFFAOYSA-N,KWILGNNWGSNMPA,BGC0001244
9364,CCMSLIB00005748333,Massbank:PR311132 DIMBOA,212.0550,0.0000,O=C1N(O)C=2C=CC(OC)=CC=2(OC1(O)),1S/C9H9NO5/c1-14-5-2-3-6-7(4-5)15-9(12)8(11)10...,GDNZNIJPBQATCZ-UHFFFAOYSA-N,GDNZNIJPBQATCZ,BGC0000810


In [ ]:
# unique_bgcs1.to_csv(os.path.join(root_gnps_path, 'gnps_mibig_unique_bgcs_from_original_json.csv'), index=False)

In [46]:
# Load the saved unique BGCs 
unique_bgcs1 = pd.read_csv(os.path.join(root_gnps_path, 'gnps_mibig_unique_bgcs_from_original_json.csv'))
print(f'shape of unique_bgcs1: {unique_bgcs1.shape}')

shape of unique_bgcs1: (434, 9)
